# Quiz 10 — cubic magnon terms and four-magnon damping (20 questions)

Work through **24_tutorial_cubic_four_magnon_damping.ipynb** first. Questions 4–6, 8, 9, 13 and 14 are hands-on.

**How to answer:** replace `...` by your answer and run the cell — a letter for multiple choice, a number or a text otherwise. `q.hint(n)`, `q.reveal(n)`, `q.score()`.
Prefer clicking? use `q.widget(n)` (needs `ipywidgets`).

In [1]:
# run this first (works from the repository's notebooks/ folder)
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # use the local package if not pip-installed
EX = os.path.abspath("../examples")
%matplotlib inline

In [2]:
from symphase.quiz import Quiz
q = Quiz('nonlinear')
print(q.title, '-', len(q), 'questions')

Quiz 10 — cubic magnon terms and four-magnon damping - 20 questions


### Helper cell for the hands-on questions

In [3]:
import math, numpy as np, spglib
import matplotlib.pyplot as plt
from symphase.magint import FourMagnonHF, MagnonSelfEnergy
from symphase.magcheck import BosonCluster, classical_expansion
from symphase.magnons import relax_spins, make_path, analyse
from symphase.tensors import read_poscar, read_incar
a = 3.0

def canted_chain(h=4.2):
    """AFM chain, XYZ exchange (Jy = 1.3), field h along −z: canted, gapped — the ED test bed"""
    J = np.diag([1.0, 1.3, 1.0]); S = np.array([1.0, 1.0])
    pairs = [(0, 1, np.array([a, 0, 0.]), J), (1, 0, np.array([a, 0, 0.]), J)]
    hfield = np.array([[0, 0, -h]] * 2)
    dirs, _, _ = relax_spins(np.array([[0, 1, 0.5], [0, -1, 0.5]]), S, [0, 1], pairs, [None, None], hfield=hfield,
                             steps=200000, tol=1e-15)
    return FourMagnonHF(np.diag([2 * a, 12, 12]), np.array([[0, 0, 0], [a, 0, 0.]]), [0, 1], dirs, S, pairs, None, hfield)

def triangular(S=0.5, G=36):
    """120° triangular AFM (√3×√3 cell of examples/vasp/Triangular_120AFM_INCAR), J = 1 meV"""
    d = "../examples/vasp/Triangular_120AFM_INCAR"
    p = read_poscar(f"{d}/POSCAR"); mags = np.array(read_incar(f"{d}/INCAR")["magmom"]).reshape(-1, 3)
    ds = spglib.get_symmetry_dataset((p["lattice"], p["frac"], [0, 0, 0]))
    m = analyse(p["lattice"], p["frac"], [0, 0, 0], ["Cr"] * 3, mags, ds.rotations, ds.translations, J=[1.0], spin=S,
                rmax=3.1, return_model=True)
    hf = FourMagnonHF.from_model(np.asarray(p["lattice"], float), np.asarray(p["frac"], float), m).solve(grid=(G, G, 1))
    return hf, MagnonSelfEnergy(hf)

def canted_square(hr, S=1.0):
    """square-lattice Heisenberg AFM (J = 1 meV) in a field h = hr·h_s along z (h_s = 8JS), √2 cell"""
    lat = np.array([[a, a, 0], [a, -a, 0], [0, 0, 12.]])
    pairs = [(0, 1, np.array(d, float), np.eye(3)) for d in ([a, 0, 0], [-a, 0, 0], [0, a, 0], [0, -a, 0])]
    s = math.sqrt(1 - hr ** 2)
    hf = FourMagnonHF(lat, np.array([[0, 0, 0], [a, 0, 0.]]), [0, 1], np.array([[s, 0, hr], [-s, 0, hr]]),
                      np.array([S, S]), pairs, None, np.array([[0, 0, -8 * hr * S]] * 2))
    return hf, MagnonSelfEnergy(hf), lat

def square_fm(J=-2.0, K=0.5, S=1.0):
    """square-lattice ferromagnet, easy axis z (gap 2KS)"""
    pairs = [(0, 0, np.array([a, 0, 0.]), J * np.eye(3)), (0, 0, np.array([0, a, 0.]), J * np.eye(3))]
    hf = FourMagnonHF(np.diag([a, a, 12.]), np.zeros((1, 3)), [0], np.array([[0, 0, 1.]]), np.array([S]), pairs,
                      [np.diag([0, 0, -K])], None)
    return hf, MagnonSelfEnergy(hf)

def ed_check(h=4.2, nmax=8):
    """exact diagonalisation (3-cell cluster) vs bubble + tadpole and four-magnon sunset; returns rows per mode"""
    hf = canted_chain(h); se = MagnonSelfEnergy(hf)
    cl = BosonCluster(hf, 3, nmax); qs = cl.kpoints(); k = qs[1]; e = se._mset(k)[0]
    H2 = cl.quadratic()
    ed3 = cl.second_order(H2, cl.operator(se.terms3), e)
    hf.update(qs, 0.0); ed4 = cl.second_order(H2, cl.operator(se.terms4) - cl.operator(cl.hf_terms()), e); hf.hf = None
    rows = []
    for n in range(2):
        _, w, ee = se.cubic_poles(k, n, 0.0, qs=qs); bub = float(np.sum(w / (e[n] - ee)).real)
        tad = se.tadpole(k, n, 0.0, ks=qs)
        _, w, ee = se.quartic_poles(k, n, 0.0, qs=qs); sun = float(np.sum(w / (e[n] - ee)).real)
        rows.append(dict(n=n, eps=e[n], ed3=ed3[n]["second"], bubble=bub, tadpole=tad, ed4=ed4[n]["second"], sunset=sun))
    return rows

def tri_velocity(se, hf, kf=0.03):
    """2S·δε/ε of the three Goldstone branches at small k (x direction): HF (first order), Re Σ³, total"""
    k = np.array([kf * 2 * np.pi / a, 0, 0])
    o = se.onshell(k, 0.0, (36, 36, 1), eta=0.01)
    dhf = hf.first_order_shift([k])[0]
    S2 = 2 * hf.S[0]
    return dict(e0=o["e0"], hf=S2 * dhf / o["e0"], re3=S2 * o["re3"] / o["e0"],
                total=S2 * (dhf + o["re3"] + o["tad"]) / o["e0"], gamma3=o["gamma3"])


### Question 1
Where do cubic (three-magnon) terms come from?

* **(a)** from phonons
* **(b)** from (S − n_i) v_i·J·S_j^⊥ (and single-ion/Zeeman terms) — non-zero only when v_i·J·e_j ≠ 0, i.e. non-collinear order
* **(c)** from the quartic HP expansion
* **(d)** from the Zeeman term in collinear magnets only

In [4]:
q.answer(1, ...)

Q1: replace the placeholder with your answer, e.g. q.answer(1, 'b')


### Question 2
At which order in 1/S does the cubic one-loop bubble contribute to the spectrum?

* **(a)** 1/S² 
* **(b)** 1/S — the same order as Hartree–Fock
* **(c)** S
* **(d)** never

In [5]:
q.answer(2, ...)

Q2: replace the placeholder with your answer, e.g. q.answer(2, 'b')


### Question 3
What is the tadpole?

* **(a)** a phonon
* **(b)** the second-order effect of linear terms from normal-ordering H₃ — the O(1/S) shift of the classical angles
* **(c)** the four-magnon sunset
* **(d)** a numerical artefact

In [6]:
q.answer(3, ...)

Q3: replace the placeholder with your answer, e.g. q.answer(3, 'b')


### Question 4
Hands-on: ed_check() — exact-diagonalisation second-order shift of mode 0 (meV, 3 decimals)?

*numerical answer*

In [7]:
q.answer(4, ...)

Q4: replace the placeholder with your answer, e.g. q.answer(4, 'b')


### Question 5
Hands-on: the tadpole contribution of mode 0 in ed_check() (meV, 3 decimals)?

*numerical answer*

In [8]:
q.answer(5, ...)

Q5: replace the placeholder with your answer, e.g. q.answer(5, 'b')


### Question 6
Hands-on: four-magnon sunset of mode 0 in ed_check() (meV, 4 decimals)?

*numerical answer*

In [9]:
q.answer(6, ...)

Q6: replace the placeholder with your answer, e.g. q.answer(6, 'b')


### Question 7
Why is H₄ − H_HF used in the exact-diagonalisation check of the sunset?

* **(a)** for speed
* **(b)** Wick: H₄ − H_HF = :H₄: + const (normal-ordered in magnon operators), whose 2nd-order shift is the sunset
* **(c)** to remove the field
* **(d)** H_HF is zero

In [10]:
q.answer(7, ...)

Q7: replace the placeholder with your answer, e.g. q.answer(7, 'b')


### Question 8
Hands-on: triangular AFM, S = 1/2: linear-spin-wave ordered moment ⟨S⟩ on the 30×30 grid (4 decimals)?

*numerical answer*

In [11]:
q.answer(8, ...)

Q8: replace the placeholder with your answer, e.g. q.answer(8, 'b')


### Question 9
Hands-on: triangular AFM — 2S·δε/ε of the c_∥ Goldstone branch at |k|a = 0.188 (3 decimals)?

*numerical answer*

In [12]:
q.answer(9, ...)

Q9: replace the placeholder with your answer, e.g. q.answer(9, 'b')


### Question 10
Hands-on: near the ±K Goldstone branches, what do the HF and cubic terms do separately?

* **(a)** both vanish
* **(b)** each is ≈ ±37ε (a spurious gap); they cancel — Goldstone modes stay gapless
* **(c)** both are positive
* **(d)** HF is zero

In [13]:
q.answer(10, ...)

Q10: replace the placeholder with your answer, e.g. q.answer(10, 'b')


### Question 11
Why is the first-order HF shift [T†H_HF T]_nn used instead of diagonalising H + H_HF?

* **(a)** it is faster
* **(b)** strict order-by-order 1/S expansion; diagonalising mixes orders and breaks Goldstone modes in non-collinear magnets
* **(c)** they always agree
* **(d)** for finite T only

In [14]:
q.answer(11, ...)

Q11: replace the placeholder with your answer, e.g. q.answer(11, 'b')


### Question 12
Hands-on: square-lattice AFM in a field — magnons decay at T = 0 …

* **(a)** at any field
* **(b)** only above h* ≈ 0.76 h_s
* **(c)** never
* **(d)** only at saturation

In [15]:
q.answer(12, ...)

Q12: replace the placeholder with your answer, e.g. q.answer(12, 'b')


### Question 13
Hands-on: square FM (J = −2 meV, K = 0.5 meV, S = 1), k = 0.4·2π/a, T = 40 K, grid 12×12, η = σ = 0.2: Γ⁴ (meV, 2 decimals)?

*numerical answer*

In [16]:
q.answer(13, ...)

Q13: replace the placeholder with your answer, e.g. q.answer(13, 'b')


### Question 14
Hands-on: the same ferromagnet at T = 0: Γ⁴ (meV)?

*numerical answer*

In [17]:
q.answer(14, ...)

Q14: replace the placeholder with your answer, e.g. q.answer(14, 'b')


### Question 15
Thermal factor of the 2→2 scattering channel X + B → C + D?

* **(a)** 1 + n_B
* **(b)** n_B(1+n_C)(1+n_D) − (1+n_B)n_C n_D
* **(c)** n_C n_D
* **(d)** 1

In [18]:
q.answer(15, ...)

Q15: replace the placeholder with your answer, e.g. q.answer(15, 'b')


### Question 16
At which order in 1/S is the four-magnon sunset?

* **(a)** 1/S
* **(b)** 1/S² — the leading damping of collinear magnets
* **(c)** S⁰
* **(d)** 1/S³

In [19]:
q.answer(16, ...)

Q16: replace the placeholder with your answer, e.g. q.answer(16, 'b')


### Question 17
Which CLI options add the cubic terms and the four-magnon damping?

* **(a)** --relax --scan
* **(b)** --hf T --cubic  /  --hf T --damping4 [--grid4 ...]
* **(c)** --spectral
* **(d)** --supercell

In [20]:
q.answer(17, ...)

Q17: replace the placeholder with your answer, e.g. q.answer(17, 'b')


### Question 18
Why are Goldstone modes exactly at their zero excluded from the on-shell results?

* **(a)** they are unstable
* **(b)** ε = 0: the on-shell 1/S expansion is singular there (the correction itself must vanish)
* **(c)** they have no vertex
* **(d)** they are phonons

In [21]:
q.answer(18, ...)

Q18: replace the placeholder with your answer, e.g. q.answer(18, 'b')


### Question 19
What does symphase.magcheck provide?

* **(a)** plots
* **(b)** independent checks: classical-energy expansion of the HP terms and exact diagonalisation of boson clusters
* **(c)** DFT inputs
* **(d)** symmetry tables

In [22]:
q.answer(19, ...)

Q19: replace the placeholder with your answer, e.g. q.answer(19, 'b')


### Question 20
Which part is NOT complete at the quartic level?

* **(a)** exchange tensors
* **(b)** single-ion anisotropy (quartic terms harmonic only)
* **(c)** DMI
* **(d)** Zeeman

In [23]:
q.answer(20, ...)

Q20: replace the placeholder with your answer, e.g. q.answer(20, 'b')


## Your score

In [24]:
q.score()

Quiz 10 — cubic magnon terms and four-magnon damping: 0/20 correct (0 attempted)
  not yet answered: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20


0